In [3]:
%pip install pandas

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 1.7 MB/s eta 0:00:07
   ------ --------------------------------- 1.8/11.3 MB 3.6 MB/s eta 0:00:03
   ------------- -------------------------- 3.9/11.3 MB 6.0 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.3 MB 7.7 MB/s eta 0:00:01
   -------------------------------- ------- 9.2/11.3 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.3 MB 8.8 MB/s eta 0:00:01
   ---------------------------------------- 11.3/11.3 MB 8.5 MB/s  0:00:01
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)

   ---------------------------------------- 0/4 [pytz]
   ---------------------------------------- 0/4 [pytz]
   -------

In [4]:
# csv_to_paddle_labels.py
import os
import shutil
import pandas as pd

NUM_DIGITS = 5  # matches your existing pipeline

def convert_split(csv_path, src_img_dir, dst_img_dir, out_label_path, num_digits=NUM_DIGITS):
    """Reads your existing CSV (image, label) and writes:
       1. A PaddleOCR label file: <relative_img_path>\t<zero_padded_label>
       2. Copies images into dst_img_dir (skip if already there)."""
    df = pd.read_csv(csv_path)
    os.makedirs(dst_img_dir, exist_ok=True)

    written, skipped = 0, 0
    with open(out_label_path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            img_name = str(row["image"])
            src_path = os.path.join(src_img_dir, img_name)
            if not os.path.exists(src_path):
                skipped += 1
                continue

            dst_path = os.path.join(dst_img_dir, img_name)
            if not os.path.exists(dst_path):
                shutil.copy2(src_path, dst_path)

            label_str = str(int(row["label"])).zfill(num_digits)
            if len(label_str) != num_digits:
                # label longer than expected digits -- flag rather than silently truncate
                print(f"WARNING: {img_name} label '{label_str}' is not {num_digits} digits, skipping.")
                skipped += 1
                continue

            # PaddleOCR label files use forward-slash relative paths
            rel_path = os.path.join(os.path.basename(dst_img_dir), img_name).replace("\\", "/")
            f.write(f"{rel_path}\t{label_str}\n")
            written += 1

    print(f"[{os.path.basename(out_label_path)}] wrote {written} lines, skipped {skipped}.")


if __name__ == "__main__":
    PROJECT_ROOT = "dataset"  # <-- change this to your actual project root
    os.makedirs(os.path.join(PROJECT_ROOT, "labels"), exist_ok=True)

    # ---- EDIT THESE to match your actual existing paths ----
    convert_split(
        csv_path="../data_set_generator_for_cnns_only/data_set/train.csv",
        src_img_dir="../data_set_generator_for_cnns_only/data_set/train",
        dst_img_dir=os.path.join(PROJECT_ROOT, "images", "train"),
        out_label_path=os.path.join(PROJECT_ROOT, "labels", "train_list.txt"),
    )
    convert_split(
        csv_path="../data_set_generator_for_cnns_only/data_set/valid.csv",
        src_img_dir="../data_set_generator_for_cnns_only/data_set/valid",
        dst_img_dir=os.path.join(PROJECT_ROOT, "images", "valid"),
        out_label_path=os.path.join(PROJECT_ROOT, "labels", "val_list.txt"),
    )
    convert_split(
        csv_path="../data_set_generator_for_cnns_only/data_set/test.csv",
        src_img_dir="../data_set_generator_for_cnns_only/data_set/test",
        dst_img_dir=os.path.join(PROJECT_ROOT, "images", "test"),
        out_label_path=os.path.join(PROJECT_ROOT, "labels", "test_list.txt"),
    )

[train_list.txt] wrote 1535 lines, skipped 0.
[val_list.txt] wrote 200 lines, skipped 0.
[test_list.txt] wrote 212 lines, skipped 0.
